# 18 — GNN 的失效机制：过平滑、过挤压与异配性

前面的课程回答了 GNN **怎样工作**；本课转而追问一个更重要的问题：**消息传递什么时候会系统性失败？** 我们不把所有失败都归咎于‘模型不够深’或‘参数没调好’，而是用可控实验区分三种机制：节点表示随深度趋同的 **过平滑（over-smoothing）**、指数增长的远程信息被压进固定宽度向量的 **过挤压（over-squashing）**，以及邻居本来就与自己语义不同的 **异配性（heterophily）**。

本课同时使用合成图、Cora 和 Chameleon。合成图负责建立因果直觉；公开图负责检查结论能否迁移到真实数据。

## 1. 学习目标与诊断框架

完成本课后，你应该能：

1. 用表示方差、Dirichlet energy 和节点对余弦相似度识别过平滑；
2. 解释为什么过平滑不等于过拟合，也不等于梯度消失；
3. 用感受野大小、最短路与远端信息敏感度解释过挤压；
4. 区分‘层数不够而看不到远端’与‘层数够但信息被挤压’；
5. 计算 edge homophily，并解释为什么低同配图违背朴素平滑假设；
6. 判断残差、归一化与图重连分别修复了哪一个环节；
7. 在 validation 上选择方案，最后才查看 test，并用多随机种子报告波动。

贯穿全课的诊断顺序是：**先看数据和图，再看表示，最后看预测指标。** 单个 accuracy 不能告诉我们失败发生在哪里。

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from crosscity.data.citation import load_planetoid
from crosscity.data.gnn_failure_modes import (
    edge_homophily, load_wikipedia_network, make_tree_retrieval_benchmark,
    rewire_by_features,
)
from crosscity.models.gnn_failure_modes import DeepGCN, propagation_trace
from crosscity.models.static_gnn import MLPNodeClassifier
from crosscity.training.node_classification import train_node_classifier
from crosscity.utils import seed_everything

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('root:', ROOT)
print('device:', DEVICE, '| torch:', torch.__version__, '| cuda:', torch.version.cuda)

## 2. 本地预览与服务器正式实验

MacBook Air 只负责确认形状、loss、反向传播和绘图链路；正式结论应在远程服务器运行。`RUN_FULL=False` 不改变任务定义，只减少树的数量、epoch、seed 和 k-NN 的邻居数。服务器上改成 `True` 后建议从头运行全部单元格。

本课的 Cora/Chameleon 都能 full-batch 训练，不依赖第 17 课的采样后端。首次运行会下载公开数据；若服务器不能联网，可在有网络的机器缓存 `data/raw/` 后上传，但不要提交数据文件。

In [ ]:
RUN_FULL = False
SEEDS = [42, 43, 44] if RUN_FULL else [42]
MAX_EPOCHS = 300 if RUN_FULL else 40
PATIENCE = 50 if RUN_FULL else 15
HIDDEN_CHANNELS = 128 if RUN_FULL else 32
TREE_COUNT = 600 if RUN_FULL else 60
TREE_DEPTH = 6 if RUN_FULL else 4
KNN_K = 10 if RUN_FULL else 5
print({'run_full': RUN_FULL, 'seeds': SEEDS, 'epochs': MAX_EPOCHS,
       'tree_count': TREE_COUNT, 'tree_depth': TREE_DEPTH})

## 3. 最小数学模型：平滑既是能力，也是风险

忽略激活函数和可学习矩阵，一层 GCN 的核心可写成

$$H^{(l+1)}=\hat D^{-1/2}\hat A\hat D^{-1/2}H^{(l)}=\tilde A H^{(l)},\qquad \hat A=A+I.$$

若图连通且传播矩阵满足常见条件，反复乘 $\tilde A$ 会抑制高频分量，让节点表示逐渐投影到由低频特征向量张成的空间。邻居变相似正是 GCN 抗噪和利用同配性的来源；但深度过大时，不同类别也难以区分。

我们同时观察三个量：

- **特征方差** $\frac1d\sum_j\operatorname{Var}_v(h_{vj})$：趋近零表示节点差异消失；
- **Dirichlet energy** $\frac1{|E|}\sum_{(u,v)\in E}\|h_u-h_v\|_2^2$：越低表示边两端越平滑；
- **随机节点对平均余弦相似度**：趋近 1 表示方向越来越一致。

单个量会受尺度影响，所以必须联合解释。

## 4. 实验 A：不训练模型，直接观察过平滑

先在环图上做最小实验。这里没有参数、optimizer 或标签，因此曲线变化不可能由过拟合或训练失败造成，只来自传播算子本身。

In [ ]:
num_ring_nodes = 80
source = torch.arange(num_ring_nodes)
target = (source + 1) % num_ring_nodes
ring_edges = torch.stack([torch.cat([source, target]), torch.cat([target, source])])
generator = torch.Generator().manual_seed(42)
ring_x = torch.randn(num_ring_nodes, 16, generator=generator)
ring_trace = pd.DataFrame(propagation_trace(ring_x, ring_edges, steps=32))
ring_trace.iloc[[0, 1, 2, 4, 8, 16, 32]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for axis, metric, title in zip(
    axes,
    ['feature_variance', 'dirichlet_energy', 'mean_pairwise_cosine'],
    ['feature variance ↓', 'Dirichlet energy ↓', 'pairwise cosine ↑'],
):
    axis.plot(ring_trace.step, ring_trace[metric], marker='o', markersize=3)
    axis.set(xlabel='propagation steps', title=title)
plt.tight_layout(); plt.show()

### 4.1 残差和归一化能做什么？

残差更新 $H^{(l+1)}=H^{(l)}+F(H^{(l)},A)$ 为原表示提供直通路径，缓解深层优化困难，也延缓信息完全被邻域平均覆盖。LayerNorm 对每个节点的通道做标准化，稳定激活尺度。它们能让深层 GCN **更容易训练**，却不能保证图中的消息是有用的：如果邻居语义相反，稳定地传播错误信息仍然是错误。

因此正式实验会同时比较 plain deep GCN 与 residual + LayerNorm；若后者改善，证据支持‘优化/表示退化是部分原因’，但不能据此宣称所有失效都已解决。

## 5. 过挤压：看得见，不代表传得过来

一个 $b$ 叉树根节点在 $L$ 跳内最多依赖约 $1+b+\cdots+b^L=O(b^L)$ 个节点，但每层仍把消息压进固定的 $d$ 维向量。这种**输入信息量指数增长、瓶颈宽度不变**的矛盾叫过挤压。它与过平滑不同：

- 过平滑问的是‘节点表示是否越来越像’；
- 过挤压问的是‘大量远程依赖能否穿过狭窄结构瓶颈’。

增加层数只能让远端进入理论感受野，不能自动扩大信息通道。图曲率、割边、最短路和有效电阻都可刻画瓶颈；本课用最直观的二叉树检索任务。每棵树只有一个叶子带 query marker，根节点要预测该叶子的随机 bit。浅层模型看不到答案；深度达到树高后能看到，却必须从大量叶子中保留唯一被标记的信息。

In [ ]:
tree_benchmark = make_tree_retrieval_benchmark(
    num_trees=TREE_COUNT, depth=TREE_DEPTH, seed=42
)
tree_data = tree_benchmark.data
shortcut_data = tree_benchmark.with_root_shortcuts()
pd.Series({
    'trees': TREE_COUNT, 'depth/root-to-leaf distance': TREE_DEPTH,
    'nodes': tree_data.num_nodes, 'original directed edge entries': tree_data.edge_index.shape[1],
    'shortcut directed edge entries': shortcut_data.edge_index.shape[1],
    'supervised train/val/test roots': tuple(int(m.sum()) for m in
        [tree_data.train_mask, tree_data.val_mask, tree_data.test_mask]),
})

### 5.1 泄漏审计与重连干预

每棵树彼此断开，train/validation/test mask 只标记根节点，且三个集合包含不同的树。非根节点的 `y` 从不进入 loss。重连只新增‘每个叶子 ↔ 本树根’的边，不读取标签、不改变特征和数据划分。它把叶到根的最短路从 `TREE_DEPTH` 降为 1，是一个有明确机制的干预；代价是根度数显著增加，真实任务中还必须考虑建图成本和伪边噪声。

In [ ]:
assert not torch.any(tree_data.train_mask & tree_data.val_mask)
assert not torch.any(tree_data.train_mask & tree_data.test_mask)
assert torch.equal(tree_data.x, shortcut_data.x)
assert torch.equal(tree_data.y, shortcut_data.y)
print('Leakage audit passed; only edge_index changed.')

### 5.2 Smoke run：形状、loss 与反向传播

先用很小的模型走一遍完整链路。smoke run 只能发现工程错误，不能用于性能结论。

In [ ]:
smoke = make_tree_retrieval_benchmark(num_trees=12, depth=2, seed=0).data.to(DEVICE)
seed_everything(0)
smoke_model = DeepGCN(3, 16, 2, num_layers=2, residual=True,
                      normalization='layer', dropout=0.0).to(DEVICE)
smoke_result = train_node_classifier(smoke_model, smoke, max_epochs=3, patience=3)
print('logits:', tuple(smoke_model(smoke.x, smoke.edge_index).shape),
      '| best epoch:', smoke_result.best_epoch,
      '| val:', round(smoke_result.validation_accuracy, 3))

In [ ]:
tree_records = []
tree_configs = [
    ('original/shallow-1', tree_data, 1, False, 'none'),
    (f'original/deep-{TREE_DEPTH}', tree_data, TREE_DEPTH, False, 'none'),
    (f'original/deep-{TREE_DEPTH}-resnorm', tree_data, TREE_DEPTH, True, 'layer'),
    ('rewired/shallow-1', shortcut_data, 1, False, 'none'),
]
for seed in SEEDS:
    for name, experiment, layers, residual, normalization in tree_configs:
        seed_everything(seed)
        experiment = experiment.to(DEVICE)
        model = DeepGCN(3, HIDDEN_CHANNELS, 2, layers, residual=residual,
                        normalization=normalization, dropout=0.2).to(DEVICE)
        started = time.perf_counter()
        result = train_node_classifier(
            model, experiment, max_epochs=MAX_EPOCHS, patience=PATIENCE,
            learning_rate=0.01, weight_decay=1e-4,
        )
        tree_records.append({
            'seed': seed, 'configuration': name, 'best_epoch': result.best_epoch,
            'validation_accuracy': result.validation_accuracy,
            'test_accuracy': result.test_accuracy,
            'seconds': time.perf_counter() - started,
        })
tree_results = pd.DataFrame(tree_records)
tree_results.groupby('configuration')[['validation_accuracy', 'test_accuracy', 'seconds']].agg(['mean', 'std'])

### 5.3 如何解释树实验

预先写下判据，避免看到结果后编故事：

1. `original/shallow-1` 接近随机水平是**感受野不足**，不是过挤压证据；
2. 原图深层模型达到树高却仍困难，才说明远端信息虽可达但难以保留；
3. `resnorm` 改善说明优化或表示退化参与失败，但若仍明显落后重连，结构瓶颈仍在；
4. `rewired/shallow-1` 改善支持最短路/瓶颈解释；若没有改善，应检查均值聚合造成的度稀释、任务难度和训练预算；
5. `RUN_FULL=False` 的小样本结果方差很大，只是预览，不可作为最终结论。

## 6. 异配性：当邻居不是相似样本

edge homophily 定义为

$$h_{edge}=\frac{1}{|E|}\sum_{(u,v)\in E}\mathbf 1[y_u=y_v].$$

$h_{edge}$ 高说明边常连接同类节点，但它不是完整的异配度量：类别不平衡时，随机连边也可能有较高同类比例；不同类别对可能具有稳定而有用的关系模式。这里把它作为首个数据审计量，而不是最终定论。

朴素 GCN 近似低通滤波，默认邻居特征应当平滑。Cora 引用图通常较同配；Chameleon 网页图更异配。异配图并不意味着‘图没有信息’，而是**相同权重的邻居平均**可能不是正确归纳偏置。关系类型、符号传播、高低频滤波或保留 ego/neighbor 分支的模型都可能更合适。

In [ ]:
datasets = {
    'Cora': load_planetoid(ROOT / 'data/raw/planetoid', 'cora'),
    'Chameleon': load_wikipedia_network(ROOT / 'data/raw/wikipedia', 'chameleon', split=0),
}
audit_rows = []
for name, data in datasets.items():
    audit_rows.append({
        'dataset': name, 'nodes': data.num_nodes, 'features': data.num_features,
        'classes': data.num_classes, 'directed edge entries': data.edge_index.shape[1],
        'edge homophily': edge_homophily(data.edge_index, data.y),
        'train': int(data.train_mask.sum()), 'validation': int(data.val_mask.sum()),
        'test': int(data.test_mask.sum()),
    })
audit = pd.DataFrame(audit_rows).set_index('dataset')
audit

## 7. 无标签图重连

我们用节点特征余弦相似度构造 k-NN 图。该操作**不读取标签**，因此比按标签连边更接近可部署干预；但特征本身若由未来信息生成，仍会泄漏，所以真实项目必须追溯特征时间。

重连不是免费午餐：它会删除原图中可能有用的异配关系，也可能把特征相似但语义不同的节点连起来。这里使用‘完全替换’而非与原边混合，是为了让干预清晰；课后可把原图和 k-NN 图作为两种关系分别建模。

In [ ]:
graph_variants = {}
for dataset_name, data in datasets.items():
    rewired = rewire_by_features(data, k=KNN_K)
    graph_variants[(dataset_name, 'observed')] = data
    graph_variants[(dataset_name, f'feature-knn-{KNN_K}')] = rewired
    print(dataset_name, '| observed h=', round(edge_homophily(data.edge_index, data.y), 3),
          '| feature-kNN h=', round(edge_homophily(rewired.edge_index, data.y), 3),
          '| kNN edges=', rewired.edge_index.shape[1])

## 8. 正式比较：失败来自深度、优化还是图？

比较对象有明确角色：

- `MLP`：完全不读图，回答节点特征单独能做到多少；
- `GCN-2`：常规浅层消息传递；
- `GCN-8-plain`：暴露深度带来的退化；
- `GCN-8-resnorm`：检查残差与 LayerNorm 能否缓解深层优化/过平滑；
- `feature-kNN`：检查失败是否主要来自观测图不符合平滑假设。

所有 checkpoint 只按 validation accuracy 选择。代码会保存 test 结果，但在下一单元只展示 validation；应在模型族、深度和重连规则冻结后，才运行最终 test 汇总单元。

In [ ]:
records = []
for dataset_name, base_data in datasets.items():
    experiment_specs = [
        ('MLP', 'observed', 0, False, 'none'),
        ('GCN-2', 'observed', 2, False, 'none'),
        ('GCN-8-plain', 'observed', 8, False, 'none'),
        ('GCN-8-resnorm', 'observed', 8, True, 'layer'),
        ('GCN-2', f'feature-knn-{KNN_K}', 2, False, 'none'),
        ('GCN-8-resnorm', f'feature-knn-{KNN_K}', 8, True, 'layer'),
    ]
    for seed in SEEDS:
        for model_name, graph_name, layers, residual, normalization in experiment_specs:
            seed_everything(seed)
            data = graph_variants[(dataset_name, graph_name)].to(DEVICE)
            if model_name == 'MLP':
                model = MLPNodeClassifier(
                    data.num_features, HIDDEN_CHANNELS, data.num_classes, dropout=0.5
                ).to(DEVICE)
            else:
                model = DeepGCN(
                    data.num_features, HIDDEN_CHANNELS, data.num_classes, layers,
                    residual=residual, normalization=normalization, dropout=0.5,
                ).to(DEVICE)
            started = time.perf_counter()
            result = train_node_classifier(
                model, data, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                learning_rate=0.01, weight_decay=5e-4,
            )
            records.append({
                'dataset': dataset_name, 'seed': seed, 'model': model_name,
                'graph': graph_name, 'best_epoch': result.best_epoch,
                'validation_accuracy': result.validation_accuracy,
                'test_accuracy': result.test_accuracy,
                'seconds': time.perf_counter() - started,
            })
results = pd.DataFrame(records)
validation_summary = (results.groupby(['dataset', 'model', 'graph'])
    [['validation_accuracy', 'seconds']].agg(['mean', 'std']))
validation_summary

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(14, 4), sharey=True)
for axis, (dataset_name, frame) in zip(np.atleast_1d(axes), results.groupby('dataset')):
    labels = frame['model'] + '\n' + frame['graph']
    plot_frame = frame.assign(configuration=labels).groupby('configuration').validation_accuracy.agg(['mean', 'std'])
    axis.bar(range(len(plot_frame)), plot_frame['mean'], yerr=plot_frame['std'].fillna(0), capsize=3)
    axis.set_xticks(range(len(plot_frame)), plot_frame.index, rotation=55, ha='right')
    axis.set(title=dataset_name, ylabel='validation accuracy', ylim=(0, 1))
plt.tight_layout(); plt.show()

## 9. 冻结选择后查看 test

运行本单元前先在实验记录中写下：每个数据集根据 validation 选择了哪个配置、选择理由是什么。不要因 test 不理想返回修改 k、深度或随机种子。多 seed 的均值描述期望表现，标准差描述稳定性；只有一个 seed 时 `std=NaN` 是正常的，不能假装它代表稳定结论。

In [ ]:
final_summary = (results.groupby(['dataset', 'model', 'graph'])
    [['validation_accuracy', 'test_accuracy', 'seconds']].agg(['mean', 'std']))
final_summary

## 10. 结果解释矩阵

请按证据组合，而不是按单个模型排名下结论：

| 观察 | 更支持的解释 | 不能直接推出 |
|---|---|---|
| 深层 plain 低于浅层，表示相似度随层数上升 | 过平滑/深层退化 | 图完全无用 |
| resnorm 恢复深层性能 | 优化或表示尺度是重要因素 | 过挤压已经消失 |
| 树上达到所需深度仍失败，捷径明显改善 | 远程结构瓶颈/过挤压 | 所有真实图都应加捷径 |
| MLP 胜过 observed GCN，且 edge homophily 低 | 朴素平滑假设不匹配 | 图没有预测信号 |
| feature-kNN 改善 | 原始边定义可能不适合当前 GCN | k-NN 是最优或无泄漏 |
| Cora 改善、Chameleon 退化 | 同一归纳偏置依赖图结构 | 实现有 bug |

还应报告运行时间。重连会改变边数，深层模型会增加激活和参数计算；小幅 accuracy 增益未必值得工程成本。远程服务器若使用 CUDA，可额外记录 `torch.cuda.max_memory_allocated()`。

## 11. 三种失效机制的边界

三者经常同时出现，但干预方向不同：

- **过平滑**：限制有效传播深度、残差/初始特征注入、规范化、解耦传播与变换；
- **过挤压**：重连、virtual node、位置/距离编码、扩大通道或使用能跨越长路径的全局机制；
- **异配性**：分离 ego 与 neighbor、学习高频/符号滤波、关系感知传播、只在可信边上传播。

关键区别是：残差能保留自己的表示，却不会凭空增加跨越图割的通信带宽；重连能缩短距离，却可能制造不符合语义的边；异配模型能利用‘不同类相连’的规律，却不能修复训练/测试泄漏。任何结构技巧都不能替代可信评价协议。

## 12. 课后练习

1. 在 Cora 上对训练后每一层 hidden representation 计算三项平滑指标；比较 plain 与 resnorm，而不只看最终 logits。
2. 把树深从 3 增加到 7，固定 hidden width，画出原图与捷径图 accuracy 随深度的曲线；再固定深度改变 width。
3. 将根节点捷径改为每棵树增加一个 virtual node，比较边数、最短路、准确率和运行时间。
4. 对 Chameleon 的 10 个官方 split 全部运行，不能只挑表现最好的 split。
5. 比较‘替换为 feature-kNN’与‘原图 + feature-kNN’；后者应建成两类关系，而不是在同一个 `edge_index` 中丢失来源。
6. 计算类别先验下随机连边的预期同配率 $\sum_c p_c^2$，用 adjusted homophily 避免类别不平衡误导。
7. 尝试 APPNP、GCNII 或适合异配图的模型，但必须保留 MLP、GCN-2 和相同数据划分。

最终实验报告应包含：配置、数据审计、validation 选择规则、多 seed 均值±标准差、test、运行时间、失败结论，以及哪些判断仍只是推测。